# Lab 5.2 — Prompt Engineering
**Module 5: Generative AI APIs & Intro to RAG**

In this lab you will:
- Apply **zero-shot** prompting for direct Nutanix ticket classification
- Improve accuracy with **few-shot** examples embedded in the prompt
- Use **chain-of-thought** (CoT) to get step-by-step AOS troubleshooting
- Set a **system prompt** persona to control tone and domain expertise
- Force **structured JSON output** for downstream parsing
- Build a reusable `PromptTemplate` class for production use

> **Instructor Note:** Prompt engineering is *software engineering for LLMs*. A well-crafted prompt is as valuable as a well-trained model. In Nutanix's AIOps context, prompt design determines whether the LLM gives a generic answer about storage or a precise answer about Stargate — the difference between a useful tool and a frustrating one.

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| google-generativeai | `google-generativeai` |
| pandas | `pandas` |

**Install all at once:**
```bash
pip install google-generativeai pandas
```

### 🔑 API Key Required

This lab calls the **Google Gemini API**. Set your key before running:

```bash
export GEMINI_API_KEY="AIza..."
```

The first cell will raise `EnvironmentError` if the key is missing.
Get a free key at: [aistudio.google.com](https://aistudio.google.com/app/apikey)

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


In [6]:
import os, json, time
import google.generativeai as genai
import pandas as pd

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise EnvironmentError('Set GEMINI_API_KEY before running this lab.')

genai.configure(api_key=GEMINI_API_KEY)
MODEL = 'gemini-3.1-flash-lite'

def call_gemini(messages, system=None, max_tokens=512, temperature=0.2, label=''):
    model_inst = genai.GenerativeModel(
        model_name=MODEL,
        system_instruction=system,
    )
    user_content = messages[-1]['content']
    response = model_inst.generate_content(
        user_content,
        generation_config=genai.GenerationConfig(
            max_output_tokens=max_tokens,
            temperature=temperature,
        )
    )
    return response.text

print('Ready ✅')

Ready ✅


## 1. Zero-Shot Prompting

**Zero-shot** means giving the model a task description and input with no examples. The model uses its pre-trained knowledge to solve the task.

> **Instructor Note:** Zero-shot is your starting point. It works surprisingly well when the task is clearly described and the categories are meaningful English words. For highly domain-specific categories (e.g. Nutanix-specific error families) zero-shot may struggle — that's when few-shot kicks in.

In [7]:
# Zero-shot: classify ticket without any examples
ZERO_SHOT_SYSTEM = """You are a Nutanix L1 support triage agent.
Classify the incoming support ticket into exactly one of these categories:
  storage | network | compute | prism | replication | performance

Respond with ONLY the category name in lowercase. No explanation, no punctuation."""

test_tickets = [
    ('Stargate crash loop after disk sda3 threw repeated IO errors',          'storage'),
    ('OVS bridge on node-2 lost connectivity after upstream switch reconfig', 'network'),
    ('CVM running at 99% memory, Cassandra service OOM killed twice today',   'compute'),
    ('Prism Central upgrade stuck at 73%, health check failing',              'prism'),
    ('Cerebro async replication lag 6 hours — double our RPO threshold',      'replication'),
    ('Random 4K write latency spiked from 1ms to 45ms after AOS upgrade',    'performance'),
]

print('=== Zero-Shot Classification ===\n')
print(f'{"Ticket":<58} {"Truth":<14} {"Predicted"}')
print('-' * 90)

zero_shot_results = []
for ticket, truth in test_tickets:
    pred = call_gemini(
        messages=[{'role': 'user', 'content': ticket}],
        system=ZERO_SHOT_SYSTEM,
        max_tokens=10,
        temperature=0.0,
    ).strip().lower()
    correct = '✅' if pred == truth else '❌'
    print(f'{ticket[:56]:<58} {truth:<14} {correct} {pred}')
    zero_shot_results.append(pred == truth)

print(f'\nZero-shot accuracy: {sum(zero_shot_results)}/{len(zero_shot_results)} = {sum(zero_shot_results)/len(zero_shot_results):.0%}')

=== Zero-Shot Classification ===

Ticket                                                     Truth          Predicted
------------------------------------------------------------------------------------------
Stargate crash loop after disk sda3 threw repeated IO er   storage        ✅ storage
OVS bridge on node-2 lost connectivity after upstream sw   network        ✅ network
CVM running at 99% memory, Cassandra service OOM killed    compute        ✅ compute
Prism Central upgrade stuck at 73%, health check failing   prism          ✅ prism
Cerebro async replication lag 6 hours — double our RPO t   replication    ✅ replication
Random 4K write latency spiked from 1ms to 45ms after AO   performance    ✅ performance

Zero-shot accuracy: 6/6 = 100%


## 2. Few-Shot Prompting

**Few-shot** includes labelled examples directly in the prompt. This anchors the model on exactly what output format and category boundaries you expect. For domain-specific tasks, 2–5 examples per class dramatically improves accuracy.

> **Instructor Note:** Choose few-shot examples that cover edge cases and ambiguous tickets. The *diversity* of examples matters more than the quantity. Three distinct examples per class beat six similar ones. Also notice how few-shot examples serve as implicit instructions — the format of your examples tells the model the expected output format.

In [8]:
FEW_SHOT_EXAMPLES = [
    ('Stargate WAL corruption detected on node-1, service crash-looping', 'storage'),
    ('NDFS datastore unmounted after SATA disk bad sector count exceeded threshold', 'storage'),
    ('LACP bond flapping every 30 seconds after ToR switch firmware upgrade', 'network'),
    ('MTU mismatch on storage VLAN causing Stargate internode packet drops', 'network'),
    ('VM live migration failing with dirty page timeout on 64GB memory guest', 'compute'),
    ('AHV host CPU ready time above 25% on overcommitted node after new VM deploy', 'compute'),
    ('Prism REST API /vms endpoint returning 504 timeout for requests over 200 VMs', 'prism'),
    ('RBAC policy not propagating to Prism Element after Prism Central RBAC edit', 'prism'),
    ('NearSync replication health degraded, 5 VMs excluded from protection domain', 'replication'),
    ('Metro availability witness VM not responding, cluster at split-brain risk', 'replication'),
    ('SSD cache hit ratio dropped from 90% to 25% after new sequential workload added', 'performance'),
    ('Erasure coding rebuild consuming all IOPS causing noisy neighbour for prod VMs', 'performance'),
]

def build_few_shot_system(examples):
    """Build a few-shot system prompt with embedded examples."""
    example_block = '\n'.join([f'Ticket: {t}\nCategory: {c}' for t, c in examples])
    return f"""You are a Nutanix L1 support triage agent.
Classify the incoming ticket into one of: storage | network | compute | prism | replication | performance
Respond with ONLY the category name in lowercase.

Examples:
{example_block}"""

FEW_SHOT_SYSTEM = build_few_shot_system(FEW_SHOT_EXAMPLES)
print(f'Few-shot prompt: {len(FEW_SHOT_SYSTEM.split())} words, ~{len(FEW_SHOT_SYSTEM)//4} tokens')

print('\n=== Few-Shot Classification ===\n')
print(f'{"Ticket":<58} {"Truth":<14} {"Zero-shot":<12} {"Few-shot"}')
print('-' * 105)

few_shot_results = []
for i, (ticket, truth) in enumerate(test_tickets):
    pred_few = call_gemini(
        messages=[{'role': 'user', 'content': ticket}],
        system=FEW_SHOT_SYSTEM,
        max_tokens=10,
        temperature=0.0,
    ).strip().lower()
    pred_zero = 'storage'  # placeholder — reuse prior results

    z_mark = '✅' if zero_shot_results[i] else '❌'
    f_mark = '✅' if pred_few == truth else '❌'
    print(f'{ticket[:56]:<58} {truth:<14} {z_mark} {"":8} {f_mark} {pred_few}')
    few_shot_results.append(pred_few == truth)

print(f'\nFew-shot accuracy: {sum(few_shot_results)}/{len(few_shot_results)} = {sum(few_shot_results)/len(few_shot_results):.0%}')

Few-shot prompt: 208 words, ~355 tokens

=== Few-Shot Classification ===

Ticket                                                     Truth          Zero-shot    Few-shot
---------------------------------------------------------------------------------------------------------
Stargate crash loop after disk sda3 threw repeated IO er   storage        ✅          ✅ storage
OVS bridge on node-2 lost connectivity after upstream sw   network        ✅          ✅ network
CVM running at 99% memory, Cassandra service OOM killed    compute        ✅          ❌ storage
Prism Central upgrade stuck at 73%, health check failing   prism          ✅          ✅ prism
Cerebro async replication lag 6 hours — double our RPO t   replication    ✅          ✅ replication
Random 4K write latency spiked from 1ms to 45ms after AO   performance    ✅          ✅ performance

Few-shot accuracy: 5/6 = 83%


## 3. Chain-of-Thought (CoT) Prompting

**Chain-of-thought** asks the model to reason step-by-step before giving a final answer. For complex troubleshooting tasks, CoT dramatically improves the quality and correctness of the output — the model "shows its work".

Format: `Let's think step by step` or a structured reasoning template.

> **Instructor Note:** CoT is most valuable when the answer requires multi-step reasoning — AOS troubleshooting is a perfect example. The tradeoff is longer output (more tokens = more cost + latency). Use CoT for high-stakes decisions (P1 incident triage) and zero/few-shot for bulk classification.

In [9]:
COT_SYSTEM = """You are a senior Nutanix L2 support engineer with 5+ years of AOS experience.
When given an incident description, reason through  the problem systematically using this structure:

1. SYMPTOMS: What the customer is observing
2. LIKELY CAUSE: Most probable root cause given the symptoms
3. DIAGNOSTIC STEPS: Exact commands to confirm the root cause (3-5 steps)
4. RESOLUTION: Step-by-step fix with specific commands
5. PREVENTION: How to avoid recurrence

Use actual Nutanix CLI commands (ncli, acli, genesis, ncc, allssh). Be specific."""

incident = """Customer reports: Stargate crash-looped 3 times in the last 2 hours on node-2 of a 4-node cluster.
AOS version is 6.5.2. Two of the four SSDs on node-2 were recently replaced after SMART failures.
The cluster is in RF2 and currently showing reduced redundancy. VMs are still running but IO latency is elevated."""

print(f'Incident:\n{incident}\n')
print('=' * 80)
print('Chain-of-Thought Response:')
print('=' * 80)

cot_response = call_gemini(
    messages=[{'role': 'user', 'content': f'Incident report:\n{incident}'}],
    system=COT_SYSTEM,
    max_tokens=700,
    temperature=0.2,
)
print(cot_response)

Incident:
Customer reports: Stargate crash-looped 3 times in the last 2 hours on node-2 of a 4-node cluster.
AOS version is 6.5.2. Two of the four SSDs on node-2 were recently replaced after SMART failures.
The cluster is in RF2 and currently showing reduced redundancy. VMs are still running but IO latency is elevated.

Chain-of-Thought Response:
This is a critical scenario. Frequent Stargate restarts in a cluster with recent disk replacements and reduced redundancy suggest that the node is struggling to rebuild data or is hitting a metadata consistency issue/I/O timeout loop due to the degraded state.

### 1. SYMPTOMS
*   **Stargate crash-looping:** Service is failing to initialize or crashing under load.
*   **Degraded State:** Cluster is in RF2 with reduced redundancy (likely due to the disk replacements).
*   **Performance Impact:** Elevated I/O latency, indicating the remaining disks are under heavy pressure or the metadata operations are stalling.

### 2. LIKELY CAUSE
The most pr

In [10]:
# Compare CoT vs Zero-shot on the same incident
ZERO_SHOT_ANSWER = call_gemini(
    messages=[{'role': 'user', 'content': f'What should I do about this Nutanix incident? {incident}'}],
    max_tokens=200,
    temperature=0.2,
)

print('Zero-shot answer (no reasoning structure):')
print('-' * 60)
print(ZERO_SHOT_ANSWER)
print()
print('Observation: CoT provides actionable commands; zero-shot gives generic advice.')

Zero-shot answer (no reasoning structure):
------------------------------------------------------------
This is a critical situation. Because you are in **RF2 with reduced redundancy**, you are currently one drive failure away from potential data loss. The `stargate` crash-loops on the node with recent SSD replacements suggest either a hardware instability, a metadata inconsistency, or a failure to rebuild data properly.

**Do not attempt to reboot the node or perform any cluster-wide maintenance until you have stabilized the situation.**

Follow these steps immediately:

### 1. Immediate Triage & Safety
*   **Stop all non-essential maintenance:** Do not perform any upgrades, node reboots, or storage migrations.
*   **Check for "Data Rebuild" status:** Log into Prism and check the **Health > Data Resiliency** dashboard. Confirm if the cluster is actively rebuilding data. If it is, **do not touch the node.**
*   **Verify CVM Memory:** Ensure the CVM on node-2 has sufficient memory alloc

## 4. System Prompts — Persona and Constraints

The **system prompt** sets the model's identity, constraints, and output style for the entire conversation. It is the most powerful lever for aligning LLM behaviour to your product requirements.

Best practices:
- State the **role** clearly ("You are a senior Nutanix L2 engineer")
- State explicit **constraints** ("Only answer questions about Nutanix AOS. Do not speculate")
- Define **output format** ("Always use the 5-section template")
- State **tone** ("Be concise and technical. No marketing language")

> **Instructor Note:** System prompts are your "product layer" on top of the raw LLM. The same base model can behave as a support engineer, a security auditor, or a capacity planner depending solely on the system prompt. This is why prompt engineering is a product skill, not just a technical skill.

In [11]:
# Test how different personas change the same answer
personas = [
    (
        'No system prompt',
        None,
    ),
    (
        'Technical L2 engineer',
        'You are a Nutanix L2 support engineer. Respond with precise technical commands. Maximum 3 sentences.',
    ),
    (
        'Executive summary',
        'You are a Nutanix solutions architect briefing a CTO. Explain in non-technical business terms. One paragraph.',
    ),
    (
        'Security-constrained',
        'You are a Nutanix KB assistant. Only answer Nutanix-specific questions. If the question is not about Nutanix infrastructure, respond: "I can only answer Nutanix-related questions."',
    ),
]

question = 'What causes Stargate to crash and how do I fix it?'

for persona_name, system_prompt in personas:
    response = call_gemini(
        messages=[{'role': 'user', 'content': question}],
        system=system_prompt,
        max_tokens=180,
        temperature=0.2,
    )
    print(f'\n--- Persona: {persona_name} ---')
    print(response.strip())


--- Persona: No system prompt ---
Because "Stargate" can refer to several different things (the *Stargate* mod for *Minecraft*, the *Stargate: Timekeepers* video game, or various fan-made projects), the causes and fixes vary.

However, most game/mod crashes follow a similar troubleshooting pattern. Here is how to diagnose and fix the issue:

---

### 1. If you are playing *Stargate: Timekeepers*
If the official RTS game is crashing:
*   **Verify Game Files:** If you are on Steam, right-click the game in your library > **Properties** > **Installed Files** > **Verify integrity of game files**. This fixes corrupted assets.
*   **Update GPU Drivers:** Ensure your NVIDIA/AMD drivers are up to date. RTS games are often sensitive to driver-level rendering issues.
*   **

--- Persona: Technical L2 engineer ---
Stargate crashes are typically caused by memory exhaustion, disk I/O latency, or metadata corruption. Run `ncli alert ls` to identify specific error codes and check `/home/nutanix/data/

In [12]:
# Test constraint enforcement
OFF_TOPIC_SYSTEM = """You are a Nutanix infrastructure KB assistant.
Only answer questions about Nutanix AOS, AHV, Prism, Cerebro, and Stargate.
If a question is unrelated to Nutanix infrastructure, respond with exactly:
OUT_OF_SCOPE: [reason in one sentence]"""

test_questions = [
    'How do I restart Stargate?',
    'What is the capital of France?',
    'Can you write a Python script to hack a database?',
    'What does Cerebro do in Nutanix?',
    'What is the best cloud provider?',
]

print('=== Constraint enforcement test ===\n')
for q in test_questions:
    resp = call_gemini(
        messages=[{'role': 'user', 'content': q}],
        system=OFF_TOPIC_SYSTEM,
        max_tokens=60,
        temperature=0.0,
    )
    label = '🔒 BLOCKED' if resp.startswith('OUT_OF_SCOPE') else '✅ ANSWERED'
    print(f'{label}  Q: {q}')
    print(f'         A: {resp.strip()[:100]}')
    print()

=== Constraint enforcement test ===

✅ ANSWERED  Q: How do I restart Stargate?
         A: To restart the Stargate service on a Nutanix node, you should perform a rolling restart to ensure cl

🔒 BLOCKED  Q: What is the capital of France?
         A: OUT_OF_SCOPE: This question is about geography and is unrelated to Nutanix infrastructure.

🔒 BLOCKED  Q: Can you write a Python script to hack a database?
         A: OUT_OF_SCOPE: I cannot assist with requests related to hacking or unauthorized access to systems.

✅ ANSWERED  Q: What does Cerebro do in Nutanix?
         A: Cerebro is the Nutanix service responsible for data replication and disaster recovery orchestration.

🔒 BLOCKED  Q: What is the best cloud provider?
         A: OUT_OF_SCOPE: This question pertains to general cloud service providers rather than Nutanix infrastr



## 5. Structured Output — Forcing JSON

LLMs can output **structured JSON** reliably when you specify the exact schema in the system prompt. This is essential for downstream parsing — the output feeds directly into databases, APIs, or the routing table from Module 4.

> **Instructor Note:** Always validate JSON output before using it in production — LLMs occasionally produce malformed JSON (missing closing brace, extra trailing comma). Use `json.loads()` in a try/except and retry on failure. In production, use Anthropic's tool use feature for guaranteed structured output.

In [13]:
JSON_SYSTEM = """You are a Nutanix ticket triage system.
Analyse the incoming support ticket and respond with ONLY a valid JSON object.
No markdown fences, no explanation — raw JSON only.

Required schema:
{
  "category": "<storage|network|compute|prism|replication|performance>",
  "priority": "<P1|P2|P3>",
  "affected_component": "<specific Nutanix service or component>",
  "immediate_action": "<one-line action for the on-call engineer>",
  "confidence": <float 0.0 to 1.0>,
  "requires_escalation": <true|false>
}

Priority guide: P1=data loss risk or cluster down, P2=degraded service, P3=informational."""

triage_tickets = [
    'Stargate has been in a crash loop for 20 minutes, cluster showing reduced redundancy, customer says VMs are unresponsive',
    'Prism Central login page is slow but accessible, REST API working fine, just cosmetic issue in the dashboard',
    'Cerebro replication lag now at 8 hours for production protection domain, RPO is 1 hour, customer panicking',
]

print('=== Structured JSON Triage Output ===\n')
parsed_results = []

for ticket in triage_tickets:
    raw = call_gemini(
        messages=[{'role': 'user', 'content': f'Ticket: {ticket}'}],
        system=JSON_SYSTEM,
        max_tokens=200,
        temperature=0.0,
    )

    # Strip markdown fences if present
    cleaned = raw.strip()
    if cleaned.startswith('```'):
        cleaned = '\n'.join(cleaned.split('\n')[1:])
    if cleaned.endswith('```'):
        cleaned = '\n'.join(cleaned.split('\n')[:-1])

    try:
        result = json.loads(cleaned)
        parsed_results.append(result)
        print(f'Ticket: "{ticket[:70]}..."' if len(ticket) > 70 else f'Ticket: "{ticket}"')
        print(json.dumps(result, indent=2))
        print()
    except json.JSONDecodeError as e:
        print(f'❌ JSON parse failed: {e}')
        print(f'Raw output: {raw}')

# Show it's parseable as a DataFrame
if parsed_results:
    df_triage = pd.DataFrame(parsed_results)
    print('=== Triage Results as DataFrame ===')
    print(df_triage[['category', 'priority', 'affected_component', 'confidence', 'requires_escalation']].to_string(index=False))

=== Structured JSON Triage Output ===

Ticket: "Stargate has been in a crash loop for 20 minutes, cluster showing redu..."
{
  "category": "storage",
  "priority": "P1",
  "affected_component": "Stargate",
  "immediate_action": "Check Stargate logs for core dumps and verify CVM memory utilization or disk latency issues.",
  "confidence": 0.98,
  "requires_escalation": true
}

Ticket: "Prism Central login page is slow but accessible, REST API working fine..."
{
  "category": "prism",
  "priority": "P3",
  "affected_component": "Prism Central UI",
  "immediate_action": "Review Prism Central service logs and browser console for frontend latency bottlenecks.",
  "confidence": 0.95,
  "requires_escalation": false
}

Ticket: "Cerebro replication lag now at 8 hours for production protection domai..."
{
  "category": "replication",
  "priority": "P1",
  "affected_component": "Cerebro",
  "immediate_action": "Investigate replication queue depth and network latency between source and target clus

## 6. Reusable PromptTemplate Class

A `PromptTemplate` separates the static structure (persona, format, examples) from the dynamic variables (ticket text, error message). This is the production pattern — store templates in a database, version them, and swap without code changes.

> **Instructor Note:** This class is a minimal version of the prompt management systems used in production LLM applications (LangChain PromptTemplate, LlamaIndex PromptTemplate). Understanding the underlying pattern means you can implement it in any language or framework without vendor lock-in.

In [14]:
class PromptTemplate:
    """Reusable prompt template with variable substitution and version tracking."""

    def __init__(self, name: str, version: str, system: str, user_template: str,
                 max_tokens: int = 512, temperature: float = 0.2):
        self.name          = name
        self.version       = version
        self.system        = system
        self.user_template = user_template  # use {var_name} placeholders
        self.max_tokens    = max_tokens
        self.temperature   = temperature

    def render(self, **kwargs) -> str:
        """Substitute variables into the user template."""
        return self.user_template.format(**kwargs)

    def run(self, model_name: str, **kwargs) -> str:
        """Render and call Gemini in one step."""
        user_msg = self.render(**kwargs)
        model_inst = genai.GenerativeModel(
            model_name=model_name,
            system_instruction=self.system,
        )
        response = model_inst.generate_content(
            user_msg,
            generation_config=genai.GenerationConfig(
                max_output_tokens=self.max_tokens,
                temperature=self.temperature,
            )
        )
        return response.text

    def __repr__(self):
        return f'PromptTemplate(name={self.name!r}, version={self.version!r})'


# ── Define reusable templates ──────────────────────────────────────────────

triage_template = PromptTemplate(
    name    = 'nutanix_triage',
    version = 'v2.1',
    system  = JSON_SYSTEM,
    user_template = 'Ticket ID: {ticket_id}\nTicket: {ticket_text}',
    max_tokens    = 200,
    temperature   = 0.0,
)

cot_rca_template = PromptTemplate(
    name    = 'nutanix_rca',
    version = 'v1.0',
    system  = COT_SYSTEM,
    user_template = 'AOS Version: {aos_version}\nNode count: {node_count}\nIncident: {incident_description}',
    max_tokens    = 600,
    temperature   = 0.2,
)

print('Templates defined:')
for t in [triage_template, cot_rca_template]:
    print(f'  {t}')

# ── Use triage template ────────────────────────────────────────────────────
print('\n=== Triage template run ===')
result_raw = triage_template.run(
    MODEL,
    ticket_id='TKT-20240115-0042',
    ticket_text='Storage latency spiked to 55ms for all VMs after disk rebuild started on node-1',
)
# Strip markdown fences if present
cleaned = result_raw.strip()
if cleaned.startswith('```'):
    cleaned = '\n'.join(cleaned.split('\n')[1:])
if cleaned.endswith('```'):
    cleaned = '\n'.join(cleaned.split('\n')[:-1])
try:
    print(json.dumps(json.loads(cleaned), indent=2))
except json.JSONDecodeError:
    print(result_raw)

Templates defined:
  PromptTemplate(name='nutanix_triage', version='v2.1')
  PromptTemplate(name='nutanix_rca', version='v1.0')

=== Triage template run ===
{
  "category": "storage",
  "priority": "P2",
  "affected_component": "Stargate",
  "immediate_action": "Check disk rebuild progress and I/O contention on node-1 via NCC and Prism.",
  "confidence": 0.95,
  "requires_escalation": false
}


## 7. Multi-Turn Conversation

The `messages` array supports alternating `user` / `assistant` turns, enabling **contextual follow-up** without repeating information. This is how you build a troubleshooting chatbot.

> **Instructor Note:** Context is the LLM's memory within a conversation. Without multi-turn, the model has no memory of previous questions — each call is independent. With multi-turn, the model can ask clarifying questions, refine its analysis, and build on prior responses. The cost scales linearly with conversation length because you pass the full history every time.

In [15]:
L2_SYSTEM = """You are a senior Nutanix L2 support engineer.
Ask clarifying questions when needed. Be concise and technical.
Remember details from earlier in the conversation."""

_chat_model = genai.GenerativeModel(
    model_name=MODEL,
    system_instruction=L2_SYSTEM,
)
chat_session = _chat_model.start_chat()

def chat(user_message: str) -> str:
    """Send a message; chat_session automatically maintains history."""
    response = chat_session.send_message(
        user_message,
        generation_config=genai.GenerationConfig(max_output_tokens=300, temperature=0.2)
    )
    return response.text

# Simulate a troubleshooting conversation
turns = [
    'Stargate is crash-looping on one of our nodes.',
    'AOS 6.5.1, 4-node cluster, happened after replacing two SSDs last night.',
    'Yes, I see this in the stargate log: WAL corruption at offset 0x800000, checksum mismatch.',
    'The WAL directory is at /home/nutanix/data/stargate-storage/wal/ — what exactly should I delete?',
]

print('=== Multi-Turn Troubleshooting Conversation ===\n')
for user_msg in turns:
    print(f'Engineer: {user_msg}')
    response = chat(user_msg)
    print(f'Gemini L2: {response.strip()}')
    print()

history = chat_session.history
print(f'Conversation turns: {len(history)}  |  Context tokens ~{sum(len(str(m)) for m in history) // 4}')

=== Multi-Turn Troubleshooting Conversation ===

Engineer: Stargate is crash-looping on one of our nodes.
Gemini L2: To begin the investigation, I need to identify the root cause of the crash. Please provide the following:

1. **Cluster Version:** What AOS version are you running?
2. **Logs:** Please run `logbay collect --aggregate=true --duration=1h` and provide the bundle. If you cannot upload, check `/home/nutanix/data/logs/stargate.FATAL` or `stargate.ERROR` on the affected node and paste the last 20 lines of the stack trace.
3. **Context:** Did this start after a specific event (e.g., node reboot, firmware upgrade, or a specific storage-heavy task)?
4. **CVM Status:** Run `cluster status` and `top` on the affected CVM. Is the CVM under high memory pressure or experiencing OOM kills?

**Immediate diagnostic step:**
Run `tail -f /home/nutanix/data/logs/stargate.out` on the affected node and observe the output for a few cycles to see if it reports a specific assertion failure or segm

## 8. Lab Summary

| Technique | When to Use | Nutanix Example |
|-----------|-------------|----------------|
| Zero-shot | Clear categories, LLM has domain knowledge | Quick ticket triage |
| Few-shot | Domain-specific labels, need to anchor format | AOS error family classification |
| Chain-of-thought | Multi-step reasoning, P1 incident analysis | Root cause investigation |
| System prompt | Every production call — sets identity + constraints | L2 engineer persona, scope limits |
| JSON output | Downstream parsing, integration with APIs | Triage → routing table → ServiceNow |
| Multi-turn | Interactive troubleshooting, follow-up questions | Support chatbot session |

> **Instructor Note:** Lab 5.3 introduces RAG — the technique that lets you inject private knowledge (Nutanix KB articles) into the LLM's context window without fine-tuning. The prompt engineering techniques from this lab are used verbatim inside the RAG pipeline: the system prompt sets the persona, the retrieved chunks go in the user message, and JSON output structures the final answer.

---
## 🎯 Challenges

### Challenge 1 — Few-Shot Quality
The few-shot system prompt includes 2 examples per category (12 total). Test whether reducing to 1 example per class or increasing to 3 examples per class changes accuracy on the 6 test tickets. What is the optimal examples-per-class count?

### Challenge 2 — JSON Retry Logic
Add a `retry` parameter to `PromptTemplate.run()` that automatically retries on `json.JSONDecodeError` up to 3 times, adding the following message to the conversation: `"Your previous response was not valid JSON. Respond with ONLY raw JSON."` Demonstrate it catches and recovers from a malformed JSON response.

### Challenge 3 — Function Calling (Structured Output)
Gemini supports **function calling** to guarantee structured output. Define a function schema matching the triage output fields and call `model.generate_content(contents, tools=[...])`. Extract the function call arguments from `response.candidates[0].content.parts[0].function_call.args`. Compare output reliability to the system-prompt JSON approach.

In [ ]:
# Challenge workspace
